# Évaluation finale des modèles

## Objectif

Ce notebook réalise l’évaluation finale des modèles Gemini et OpenAI sur les questions médicales MCQU du split `test`.

Le prompt sélectionné dans le notebook précédent est utilisé de manière identique pour tous les modèles. Le split `test` n’a pas été utilisé pendant la conception et la sélection du prompt.

Les résultats enregistrés permettront de mesurer :

- l’exactitude des réponses ;
- la validité du format ;
- la latence ;
- le taux d’erreurs techniques ;
- la confiance déclarée ;
- les désaccords entre modèles ;
- les erreurs produites avec une confiance élevée.

In [25]:
from pathlib import Path
import os
import re
import time

import pandas as pd
from dotenv import load_dotenv

In [26]:
from pathlib import Path
import sys

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [27]:
# importation des fonctions crées à partir du notebook 03_evaluation_finale stockées dans le dossier src
from src.prompt import prepare_sample
from src.llm_inference import call_llm, run_single_experiment
from src.evaluation import build_model_comparison, get_model_errors, save_csv, save_parquet
from src.experiment_runner import run_experiment_batch



In [28]:

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "final_evaluation"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRAIN_DATA_PATH = (
    PROCESSED_DIR
    / "benchmark_mcqu_train.parquet"
)

In [29]:
df_trn = pd.read_parquet(
    TRAIN_DATA_PATH
)

In [30]:
RANDOM_SEED = 42
PROMPT_SAMPLE_SIZE = 7000
df_sample = (
    df_trn.sample(
        n=PROMPT_SAMPLE_SIZE,
        random_state=RANDOM_SEED
    )
)

In [31]:
# Chargement de la clé API du fichier .env
from google import genai
from openai import OpenAI
env_path = Path.cwd()/ ".env"
load_dotenv(env_path)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Initialisation des clients Gemni et OpenAI
gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)
llm_name_1 = "gemini"
model_name_1="gemini-3.6-flash"

openai_client = OpenAI(
    api_key=OPENAI_API_KEY
)
llm_name_2 = "openai"
model_name_2="gpt-5.4-mini-2026-03-17"

In [32]:
df_prepared_sample = prepare_sample(df_sample)

In [33]:
display(df_sample)

,sample_id,id,configuration,split,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,choices,choices_text,reference_letter,reference_answer,medical_subject,question_type,task,question_context
9571,mcqu_train_13934,13934,mcqu,train,,"Dans un test de dépistage d'une maladie, la pr...",La spécifité,La sensibilité,La prévalence,L'incidence,La valeur prédictive positive,"{'A': 'La spécifité', 'B': 'La sensibilité', '...",A. La spécifité\nB. La sensibilité\nC. La prév...,B,La sensibilité,Epidemiology,Understanding,QCU,Question :\nDans un test de dépistage d'une ma...
9502,mcqu_train_18036,18036,mcqu,train,"Une femme de 47 ans, nulligeste, jusque là nor...","Parmi les examens complémentaires suivants, qu...",Frottis cervico-vaginaux de cytodétection,Biopsie d'endomètre,Radiographie de l'abdomen sans préparation,Numération - formule sanguine et taux d'hémogl...,Hystérosalpingographie,{'A': 'Frottis cervico-vaginaux de cytodétecti...,A. Frottis cervico-vaginaux de cytodétection\n...,D,Numération - formule sanguine et taux d'hémogl...,Gynecology and Obstetrics,Reasoning,QCU,"Cas clinique :\nUne femme de 47 ans, nulligest..."
3379,mcqu_train_9861,9861,mcqu,train,,Lors de la prescription d'un collyre mydriatiq...,Prendre le tonus oculaire,Vérifier l'état de l'angle camérulaire,Vérifier l'état de la rétine périphérique,Vérifier l'état de la papille,Vérifier l'état de la macula,"{'A': 'Prendre le tonus oculaire', 'B': 'Vérif...",A. Prendre le tonus oculaire\nB. Vérifier l'ét...,B,Vérifier l'état de l'angle camérulaire,Ophthalmology,Understanding,QCU,Question :\nLors de la prescription d'un colly...
5407,mcqu_train_1065,1065,mcqu,train,,Parmi les affirmations suivantes la(lesquelles...,1+2+3,1+3,2+4,4,1+2+3+4,"{'A': '1+2+3', 'B': '1+3', 'C': '2+4', 'D': '4...",A. 1+2+3\nB. 1+3\nC. 2+4\nD. 4\nE. 1+2+3+4,E,1+2+3+4,Cardiology,Understanding,QCU,Question :\nParmi les affirmations suivantes l...
7815,mcqu_train_27194,27194,mcqu,train,"Patiente âgée de 62 ans, sans antécédents path...",Le diagnostic le plus probable est : (Cocher l...,Hypothyroïdie primaire post ménopausique,Hypothyroïdie primaire par thyroïdite de Hashi...,Hypothyroïdie primaire par thyroïdite subaigüe,Hypothyroïdie primaire par thyroïdite de Riedel,Hypothyroïdie primaire par thyroïdite atrophique,{'A': 'Hypothyroïdie primaire post ménopausiqu...,A. Hypothyroïdie primaire post ménopausique\nB...,B,Hypothyroïdie primaire par thyroïdite de Hashi...,Endocrinology and Metabolism,Reasoning,QCU,"Cas clinique :\nPatiente âgée de 62 ans, sans ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8828,mcqu_train_17727,17727,mcqu,train,,Quelle est la dimension moyenne du diamètre pr...,15 cm,"12,5 cm","10,5 cm","13,5 cm","9,5 cm","{'A': '15 cm', 'B': '12,5 cm', 'C': '10,5 cm',...","A. 15 cm\nB. 12,5 cm\nC. 10,5 cm\nD. 13,5 cm\n...",C,"10,5 cm",Gynecology and Obstetrics,Understanding,QCU,Question :\nQuelle est la dimension moyenne du...
7896,mcqu_train_21510,21510,mcqu,train,,(cochez la réponse juste) La prévention dans l...,Demander a l'asthmatique de ne faire aucun effort,Demander a l'asthmatique de limiter les effort...,Privilégier l'effort en air sec plutôt qu'en a...,Prendre une bouffée corticoïde inhalé 10' avan...,Prendre une bouffée bêta-mimétique 10' avant u...,{'A': 'Demander a l'asthmatique de ne faire au...,A. Demander a l'asthmatique de ne faire aucun ...,E,Prendre une bouffée bêta-mimétique 10' avant u...,Pulmonology,Understanding,QCU,Question :\n(cochez la réponse juste) La préve...
3705,mcqu_train_5828,5828,mcqu,train,,Une femme de 50 ans vient consulter pour doule...,Ostéonécrose de la tête fémorale droite,Chondro-calcinose,Coxarthrose postérieure,Coxarthrose sur dysplasie subluxante,Algodystrophie,{'A': 'Ostéonécrose de la tête fémorale droite...,A. Ostéonécrose de la tête fémorale droite\nB....,D,Coxarthrose sur dysplasie subluxante,Rheumatology,Reasoning,QCU,Question :\nUne femme de 50 ans vient consulte...
2021,mcqu_train_2

In [34]:
GEMINI_BENCHMARK_PATH = (
    RESULTS_DIR / "gemini_prompt_v3_benchmark.parquet"
)

#df_gemini_benchmark = run_experiment_batch(
#    experiments=df_prepared_sample,
#    llm_name=llm_name_1,
#    runner=run_single_experiment,
#    model_name=model_name_1,
#    llm_client=gemini_client,
#    output_path=GEMINI_BENCHMARK_PATH,
#    pause_seconds=1,
#)

In [35]:
OPENAI_BENCHMARK_PATH = (
    RESULTS_DIR / "openai_prompt_v3_benchmark.parquet"
)

#df_openai_benchmark = run_experiment_batch(
#    experiments=df_prepared_sample,
#    llm_name=llm_name_2,
#    runner=run_single_experiment,
#    model_name=model_name_2,
#    llm_client=openai_client,
#    output_path=OPENAI_BENCHMARK_PATH,
#    pause_seconds=1,
#)

## Les analyses et métriques

In [36]:
df_openai_results = pd.read_parquet(OPENAI_BENCHMARK_PATH)
df_gemini_results = pd.read_parquet(GEMINI_BENCHMARK_PATH)
nbr_response = len(df_gemini_results)


In [37]:
# on supprime les lignes où "predicted_letter" = "None"
df_gemini_results = df_gemini_results.dropna(
    subset=["predicted_letter"]
).reset_index(drop=True)

In [38]:
print("nombre de lignes sans réponse de gémini : ", nbr_response - len(df_gemini_results))

nombre de lignes sans réponse de gémini :  1226


In [39]:
# on conserve uniquement les identité commune au 2 dataframe
common_sample_ids = set(
    df_gemini_results["sample_id"]
) & set(
    df_openai_results["sample_id"]
)

df_gemini_results = df_gemini_results[
    df_gemini_results["sample_id"].isin(common_sample_ids)
].reset_index(drop=True)

df_openai_results = df_openai_results[
    df_openai_results["sample_id"].isin(common_sample_ids)
].reset_index(drop=True)

In [40]:
print(f"nombre de réponse conservé de chat : ", len(df_openai_results))
print(f"nombre de réponse conservé de gémini : ", len(df_gemini_results))

nombre de réponse conservé de chat :  3791
nombre de réponse conservé de gémini :  3791


In [41]:
df_model_comparison_details, df_model_disagreements = build_model_comparison(df_gemini_results, df_openai_results, df_sample)
print("Réponses de gémini et chat dans un même Dataframe")
display(df_model_comparison_details)


Réponses de gémini et chat dans un même Dataframe


,sample_id,prompt_version,reference_letter,gemini_response,gemini_raw_response,gemini_is_correct,gemini_format_valid,gemini_justification,gemini_confidence,gemini_generation_error,openai_response,openai_raw_response,openai_is_correct,openai_format_valid,openai_justification,openai_confidence,openai_generation_error,question_context
0,mcqu_train_13934,prompt_v3,B,B,Réponse : B\nJustification : La sensibilité d'...,True,True,La sensibilité d'un test de dépistage correspo...,100.0,None,B,Réponse : B\nJustification : La sensibilité es...,True,True,La sensibilité est la capacité d’un test à êtr...,99,None,Question :\nDans un test de dépistage d'une ma...
1,mcqu_train_18036,prompt_v3,D,D,Réponse : D\nJustification : La patiente prése...,True,True,La patiente présente des métrorragies persista...,100.0,None,D,Réponse : D \nJustification : Il s’agit de mé...,True,True,Il s’agit de métrorragies persistantes chez un...,82,None,"Cas clinique :\nUne femme de 47 ans, nulligest..."
2,mcqu_train_9861,prompt_v3,B,B,Réponse : B\nJustification : L'instillation d'...,True,True,L'instillation d'un collyre mydriatique entraî...,100.0,None,B,Réponse : B \nJustification : Un collyre mydr...,True,True,Un collyre mydriatique peut déclencher ou aggr...,98,None,Question :\nLors de la prescription d'un colly...
3,mcqu_train_1065,prompt_v3,E,E,Réponse : E\nJustification : Toutes les affirm...,True,True,Toutes les affirmations sont exactes : 1) L'ap...,100.0,None,A,Réponse : A\nJustification : L’ischémie aiguë ...,False,True,L’ischémie aiguë est d’autant plus grave qu’el...,78,None,Question :\nParmi les affirmations suivantes l...
4,mcqu_train_27194,prompt_v3,B,B,Réponse : B\nJustification : Le tableau cliniq...,True,True,Le tableau clinique associe un syndrome de hyp...,95.0,None,B,Réponse : B \nJustification : Le tableau est ...,True,True,Le tableau est celui d’une hypothyroïdie franc...,92,None,"Cas clinique :\nPatiente âgée de 62 ans, sans ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3786,mcqu_train_588,prompt_v3,B,B,Réponse : B\nJustification : Un proto-oncogène...,True,True,Un proto-oncogène est un gène cellulaire norma...,100.0,None,B,Réponse : B \nJustification : Un proto-oncogè...,True,True,Un proto-oncogène est un gène normal présent d...,98,None,Question :\nUn proto-oncogène :\n\nProposition...
3787,mcqu_train_3475,prompt_v3,B,B,Réponse : B\nJustification : La réaction de cu...,True,True,La réaction de culture mixte lymphocytaire (CM...,100.0,None,B,Réponse : B\nJustification : La culture mixte ...,True,True,La culture mixte lymphocytaire met en évidence...,98,None,Question :\nQuelles sont les cellules qui répo...
3788,mcqu_train_12382,prompt_v3,D,D,Réponse : D\nJustification : L'intoxication ai...,True,True,L'intoxication aiguë par les morphiniques se c...,100.0,None,D,Réponse : D\nJustification : L’intoxication pa...,True,True,L’intoxication par les morphiniques provoque c...,98,None,Question :\nL'intoxication par les morphinique...
3789,mcqu_train_18662,prompt_v3,C,C,Réponse : C\nJustification : Le cromoglycate d...,True,True,Le cromoglycate de sodium (Lomudal°) est un an...,100.0,None,C,Réponse : C \nJustification : Le cromoglycate...,True,True,Le cromoglycate de sodium est un antiallergiqu...,98,None,Question :\nLe cromoglycate de sodium (Lomudal...


In [42]:
print("Comparaison des erreurs de réponse des 2 LLM")
display(df_model_disagreements)

Comparaison des erreurs de réponse des 2 LLM


,sample_id,prompt_version,question_context,reference_letter,gemini_response,gemini_is_correct,gemini_format_valid,gemini_justification,gemini_confidence,openai_response,openai_is_correct,openai_format_valid,openai_justification,openai_confidence
0,mcqu_train_1065,prompt_v3,Question :\nParmi les affirmations suivantes l...,E,E,True,True,Toutes les affirmations sont exactes : 1) L'ap...,100.0,A,False,True,L’ischémie aiguë est d’autant plus grave qu’el...,78
1,mcqu_train_14642,prompt_v3,Question :\nQuelle est la concentration en pro...,C,C,True,True,"Le lait maternel mature contient environ 1,0 à...",95.0,A,False,True,Le lait maternel nature contient en moyenne en...,98
2,mcqu_train_24386,prompt_v3,"Cas clinique :\nMr. R.B. demeurant à Alger, âg...",D,D,True,True,L'association de troubles du transit chronique...,100.0,A,False,True,Les douleurs abdominales chroniques avec diarr...,88
3,mcqu_train_21866,prompt_v3,"Cas clinique :\nUn homme âgé de 32 ans, origin...",D,D,True,True,La présence d'une fièvre avec frissons et d'un...,95.0,B,False,True,Le tableau évoque une infection biliaire compl...,82
4,mcqu_train_8311,prompt_v3,Cas clinique :\nUn patient de 45 ans est attei...,D,D,True,True,"Chez un homme de 45 ans, la valeur normale de ...",95.0,C,False,True,"Avant la binéphrectomie, l’anémie est essentie...",88
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
459,mcqu_train_25805,prompt_v3,"Cas clinique :\nMadame F.Z. 24 ans, à déjà acc...",C,C,True,True,La révision utérine n'est jamais systématique ...,95.0,B,False,True,"Après un accouchement prématuré, on réalise sy...",93
460,mcqu_train_6100,prompt_v3,Cas clinique :\nUn garçon de cinq ans en parfa...,B,C,False,True,Chez un enfant de 5 ans présentant une boiteri...,95.0,B,True,True,"Chez cet enfant de 5 ans, une boiterie aiguë a...",72
461,mcqu_train_24849,prompt_v3,"Question :\nEn cas de septicémie, l'antibiothé...",B,B,True,True,Le traitement antibiotique d'une septicémie (i...,95.0,D,False,True,"En cas de septicémie, l’antibiothérapie doit ê...",78
462,mcqu_train_21161,prompt_v3,Cas clinique :\nUn homme âgé de 65 ans consult...,B,D,False,True,L'occlusion de la veine centrale de la rétine ...,95.0,A,False,True,Une occlusion de la veine centrale de la rétin...,78


In [43]:
df_model_errors = get_model_errors(
    df_gemini_results,
    df_openai_results,
    df_sample,
)

In [ ]:
number_distinct_ids = df_model_errors["sample_id"].nunique()
number_duplicate_ids = df_model_errors["sample_id"].duplicated().sum()
number_openai = (df_model_errors["model_name"] == "OpenAI").sum()
number_gemini = (df_model_errors["model_name"] == "Gemini").sum()


top_3_errors_by_model = (
    df_model_errors
    .groupby(
        ["model_name", "medical_subject"]
    )
    .size()
    .reset_index(name="number_errors")
    .sort_values(
        ["model_name", "number_errors"],
        ascending=[True, False],
    )
    .groupby("model_name")
    .head(3)
    .reset_index(drop=True)
)

,model_name,medical_subject,number_errors
0,Gemini,Hepato-Gastroenterology,28
1,Gemini,Endocrinology and Metabolism,25
2,Gemini,Gynecology and Obstetrics,20
3,OpenAI,Hepato-Gastroenterology,39
4,OpenAI,Cardiology,37
5,OpenAI,Gynecology and Obstetrics,35


In [51]:
print(len(df_model_errors), " réponses incorrectes")
print(f"\n")
print(number_distinct_ids, " questions distinctes")
print(number_duplicate_ids, " questions ratées par les deux modèles")
print(f"\n")
print(number_gemini, " erreurs de Gemini")
print(number_openai, " erreurs d'OpenAI")

display(top_3_errors_by_model)

display(df_model_errors)


813  réponses incorrectes


616  questions distinctes
197  questions ratées par les deux modèles


284  erreurs de Gemini
529  erreurs d'OpenAI


,model_name,medical_subject,number_errors
0,Gemini,Hepato-Gastroenterology,28
1,Gemini,Endocrinology and Metabolism,25
2,Gemini,Gynecology and Obstetrics,20
3,OpenAI,Hepato-Gastroenterology,39
4,OpenAI,Cardiology,37
5,OpenAI,Gynecology and Obstetrics,35


,model_name,sample_id,medical_subject,question_context,reference_letter,model_response,is_correct,justification,confidence
0,Gemini,mcqu_train_23245,Orthopedics,"Cas clinique :\nUn jeune homme de 25 ans, vict...",E,B,False,L'attitude vicieuse du membre inférieur droit ...,95.0
1,Gemini,mcqu_train_11747,Infectious Diseases,Cas clinique :\nVous êtes amené à voir à votre...,B,A,False,Devant une angine streptococcique (à streptoco...,95.0
2,Gemini,mcqu_train_7689,Nephro-Urology,Question :\nUne insuffisance rénale aiguë par ...,A,B,False,L'insuffisance rénale aiguë par obstacle est p...,100.0
3,Gemini,mcqu_train_18793,Pulmonology,Cas clinique :\nUn homme de 40 ans a ressenti ...,B,C,False,Chez un patient de 40 ans (ou de plus de 40 an...,90.0
4,Gemini,mcqu_train_12851,Pulmonology,"Cas clinique :\nMonsieur M., manoeuvre de son ...",C,A,False,L'image décrit une opacité en bande d'environ ...,90.0
...,...,...,...,...,...,...,...,...,...
808,OpenAI,mcqu_train_948,Cardiology,Question :\nQuelle est la dose de xylocaïne à ...,C,B,False,La xylocaïne (lidocaïne) utilisée en préventio...,88.0
809,OpenAI,mcqu_train_25805,Gynecology and Obstetrics,"Cas clinique :\nMadame F.Z. 24 ans, à déjà acc...",C,B,False,"Après un accouchement prématuré, on réalise sy...",93.0
810,OpenAI,mcqu_train_24849,Microbiology,"Question :\nEn cas de septicémie, l'antibiothé...",B,D,False,"En cas de septicémie, l’antibiothérapie doit ê...",78.0
811,OpenAI,mcqu_train_21161,Ophthalmology,Cas clinique :\nUn homme âgé de 65 ans consult...,B,A,False,Une occlusion de la veine centrale de la rétin...,78.0


In [49]:
save_csv(df_model_errors, RESULTS_DIR, "erreur_LLM.csv")
save_parquet(df_model_errors, RESULTS_DIR, "erreur_LLM.parquet")

WindowsPath('c:/Users/MANEL/Dropbox/projet_evaluation_LLM_medicale/final_evaluation/erreur_LLM.parquet')

In [47]:
from sklearn.metrics import f1_score

def calculate_accuracy_metrics(df, string):
    valid_results = df["is_correct"].dropna()
    name = string,
    total_experiments = len(df)
    valid_experiments = len(valid_results)
    correct_answers = valid_results.sum()
    incorrect_answers = valid_experiments - correct_answers

    accuracy = valid_results.mean()

    return pd.DataFrame(
        [{
            "model":name,
            "total_experiments": total_experiments,
            "valid_experiments": valid_experiments,
            "correct_answers": int(correct_answers),
            "incorrect_answers": int(incorrect_answers),
            "accuracy": accuracy,
        }]
    )


def calculate_f1_score(df):
    valid_results = df.dropna(
        subset=[
            "reference_letter",
            "predicted_letter",
        ]
    )

    score = f1_score(
        valid_results["reference_letter"],
        valid_results["predicted_letter"],
        labels=["A", "B", "C", "D", "E"],
        average="macro",
        zero_division=0,
    )

    return score

In [48]:
precision_gemini = calculate_accuracy_metrics(df_gemini_results, "Gemini")
precision_openai = calculate_accuracy_metrics(df_openai_results, "OpenAI")


f1_gemini = calculate_f1_score(df_gemini_results)
f1_openai = calculate_f1_score(df_openai_results)


print(f"Score F1 gemini : {f1_gemini:.3f}")
print(f"Score F1 openai : {f1_openai:.3f}")
display(precision_gemini)
display(precision_openai)

Score F1 gemini : 0.921
Score F1 openai : 0.855


,model,total_experiments,valid_experiments,correct_answers,incorrect_answers,accuracy
0,"(Gemini,)",3791,3791,3507,284,0.925086


,model,total_experiments,valid_experiments,correct_answers,incorrect_answers,accuracy
0,"(OpenAI,)",3791,3791,3262,529,0.860459
